# 02. Data Cleaning

## Objective

Clean the raw customer churn dataset while preserving valid customer
information and ensuring that the resulting dataset is suitable for
exploratory data analysis and machine learning.

## Cleaning Steps

1. Load raw dataset
2. Inspect duplicates
3. Identify invalid values
4. Convert invalid values to missing values
5. Handle missing numerical values
6. Handle missing categorical values
7. Validate the cleaned dataset
8. Export the processed dataset

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "D:/Desktop/Projects/customer-churn-analytics/data/raw/ecommerce_customer_churn_dataset.csv"
)

df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,17.9,9.0,4.0,16.3,20.8,1.0,953.33,2278.0,0,Q1
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,42.8,7.0,3.0,NaN,23.3,3.0,1067.47,3028.0,0,Q4
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0.0,4.0,1.0,NaN,8.8,NaN,1289.75,2317.0,0,Q4
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,41.4,2.0,5.0,85.9,31.0,3.0,2340.92,2674.0,0,Q1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,NaN,19.1,9.0,...,37.9,1.0,11.0,83.0,50.4,4.0,3041.29,5354.0,0,Q4


In [5]:
clean_df = df.copy()

In [6]:
duplicate_count = clean_df.duplicated().sum()

print("Duplicate records:", duplicate_count)

Duplicate records: 0


In [7]:
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", clean_df.shape)

Shape after removing duplicates: (50000, 25)


In [8]:
invalid_age = (
    (clean_df["Age"] < 18) |
    (clean_df["Age"] > 100)
)

print("Invalid Age:", invalid_age.sum())

Invalid Age: 50


In [9]:
clean_df.loc[invalid_age, "Age"] = np.nan

In [10]:
invalid_purchases = clean_df["Total_Purchases"] < 0

print(
    "Invalid Total_Purchases:",
    invalid_purchases.sum()
)

Invalid Total_Purchases: 40


In [11]:
clean_df.loc[
    invalid_purchases,
    "Total_Purchases"
] = np.nan

In [12]:
rate_columns = [
    "Cart_Abandonment_Rate",
    "Discount_Usage_Rate",
    "Returns_Rate",
    "Email_Open_Rate"
]

In [13]:
for col in rate_columns:
    invalid = (
        (clean_df[col] < 0) |
        (clean_df[col] > 100)
    )

    print(
        col,
        "invalid:",
        invalid.sum()
    )

Cart_Abandonment_Rate invalid: 30
Discount_Usage_Rate invalid: 207
Returns_Rate invalid: 0
Email_Open_Rate invalid: 0


In [14]:
for col in rate_columns:
    invalid = (
        (clean_df[col] < 0) |
        (clean_df[col] > 100)
    )

    clean_df.loc[invalid, col] = np.nan

In [15]:
missing_before = clean_df.isnull().sum()

missing_before[missing_before > 0]

Age                              2545
Session_Duration_Avg             3399
Pages_Per_Session                3000
Cart_Abandonment_Rate              30
Wishlist_Items                   4000
Total_Purchases                    40
Days_Since_Last_Purchase         3000
Discount_Usage_Rate              3707
Returns_Rate                     4491
Email_Open_Rate                  2528
Customer_Service_Calls            168
Product_Reviews_Written          3500
Social_Media_Engagement_Score    6000
Mobile_App_Usage                 5000
Payment_Method_Diversity         2500
Credit_Balance                   5500
dtype: int64

In [16]:
numerical_cols = clean_df.select_dtypes(
    include=np.number
).columns.tolist()

In [17]:
numerical_features = [
    col for col in numerical_cols
    if col != "Churned"
]

In [18]:
for col in numerical_features:
    clean_df[col] = clean_df[col].fillna(
        clean_df[col].median()
    )

In [20]:
categorical_cols = clean_df.select_dtypes(
    include="object"
).columns.tolist()

categorical_cols

C:\Users\PC\AppData\Local\Temp\ipykernel_6456\2278269423.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = clean_df.select_dtypes(


['Gender', 'Country', 'City', 'Signup_Quarter']

In [21]:
for col in categorical_cols:
    clean_df[col] = clean_df[col].fillna(
        clean_df[col].mode()[0]
    )

In [22]:
missing_after = clean_df.isnull().sum()

missing_after[missing_after > 0]

Series([], dtype: int64)

In [27]:
print(
    "Age:",
    clean_df["Age"].min(),
    clean_df["Age"].max()
)
print(
    "Total_Purchases:",
    clean_df["Total_Purchases"].min(),
    clean_df["Total_Purchases"].max()
)
for col in rate_columns:
    print(
        col,
        "min =", clean_df[col].min(),
        "max =", clean_df[col].max()
    )

Age: 18.0 75.0
Total_Purchases: 0.0 128.70000000000002
Cart_Abandonment_Rate min = 0.0 max = 100.0
Discount_Usage_Rate min = 0.24 max = 99.96
Returns_Rate min = 0.0 max = 99.6157338622138
Email_Open_Rate min = 0.0 max = 91.7


In [28]:
clean_df["Churned"].value_counts()

Churned
0    35550
1    14450
Name: count, dtype: int64

In [29]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            50000 non-null  float64
 1   Gender                         50000 non-null  str    
 2   Country                        50000 non-null  str    
 3   City                           50000 non-null  str    
 4   Membership_Years               50000 non-null  float64
 5   Login_Frequency                50000 non-null  float64
 6   Session_Duration_Avg           50000 non-null  float64
 7   Pages_Per_Session              50000 non-null  float64
 8   Cart_Abandonment_Rate          50000 non-null  float64
 9   Wishlist_Items                 50000 non-null  float64
 10  Total_Purchases                50000 non-null  float64
 11  Average_Order_Value            50000 non-null  float64
 12  Days_Since_Last_Purchase       50000 non-null  float64
 1

In [31]:
clean_df.duplicated().sum()
clean_df.shape

(50000, 25)

In [32]:
quality_report = pd.DataFrame({
    "Feature": clean_df.columns,
    "Missing_Count": clean_df.isnull().sum().values,
    "Data_Type": clean_df.dtypes.astype(str).values
})

quality_report

,Feature,Missing_Count,Data_Type
0,Age,0,float64
1,Gender,0,str
2,Country,0,str
3,City,0,str
4,Membership_Years,0,float64
5,Login_Frequency,0,float64
6,Session_Duration_Avg,0,float64
7,Pages_Per_Session,0,float64
8,Cart_Abandonment_Rate,0,float64
9,Wishlist_Items,0,float64


In [33]:
quality_report[
    quality_report["Missing_Count"] > 0
]

,Feature,Missing_Count,Data_Type


In [34]:
clean_df.to_csv(
    "D:/Desktop/Projects/customer-churn-analytics/data/processed/customer_churn_clean.csv",
    index=False
)

Mục đích chính:

Xử lý các vấn đề phát hiện trong Data Understanding, tạo ra một dataset sạch và sẵn sàng cho các bước phân tích tiếp theo.

Các nội dung đã thực hiện:

Đọc dữ liệu gốc.
Tạo bản sao dữ liệu để thực hiện cleaning.
Kiểm tra và loại bỏ duplicate.
Xác định giá trị không hợp lệ.
Chuyển giá trị không hợp lệ thành NaN.
Xử lý missing values.
Imputation cho biến số bằng median.
Imputation cho biến phân loại bằng mode.
Kiểm tra lại dữ liệu sau cleaning.
Kiểm tra lại target Churned.
Xuất dataset sạch sang thư mục processed.